## Split Overmerged Authors — Phase 2: Different First Name, Same Last Name

Identifies work_authors where the parsed **last name** of `raw_author_name` matches the author profile's parsed last name, but the **first names** are clearly different.

**Safety filters:**
- Requires both first/last names > 2 chars
- No substring overlap on first name in either direction (raw or hyphen-stripped)
- Excludes CJK-source `raw_author_name` (Hangul/Hiragana/Katakana/Han) — high romanization FP rate
- Excludes CJK-source profile `full_name`
- Excludes periods on either parsed first name (initials like "Th.")
- Excludes title/degree tokens as first name (md, lord, sc., ltcol, &na, etc.)
- **Per-row ORCID exclusion** — drop the row only when the work's `raw_orcid` matches the profile's ORCID (replaces the old profile-level `ap.orcid IS NULL` filter so we can split outlier works on ORCID-anchored profiles)
- **Minority-cluster guard** — only emit candidates where the work's first-name cluster is ≤ 30% of the profile's total works. Protects the "main" identity of large profiles from getting carved up.

**Design decision — no co-author overlap filter.** A co-author overlap filter was evaluated and dropped. It would have cut the FP rate from ~11% to ~7%, but at the cost of missing ~270 true overmerges for every ~58 false positives avoided (~4.7x recall cost). The Phase 1 playbook (ship aggressive, rollback discovered systematic FP classes via the audit log) is preferred instead — see `oxjobs/working/split-authors-name-parser/eval/findings.md`.

Evaluation on 100 stratified authors (1,268 candidate work_authors) using the `aer-gold-standard` Sonnet judge: **~10.7% per-candidate FP rate**. With the per-row ORCID + 30% cluster guard, expected ~2.2M splits at this iteration.

### Cell 1: Build candidates table

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.overmerge_split_candidates_phase2 AS
WITH work_parsed AS (
  SELECT wa.work_id, wa.author_sequence, wa.author_id, wa.raw_author_name,
    an_w.parsed_name.last AS work_last,
    an_w.parsed_name.first AS work_first,
    -- Cluster size: how many works under this author share this first name.
    -- Computed BEFORE the candidate WHERE filters so it reflects the profile's
    -- overall first-name distribution, not just candidate-eligible rows.
    COUNT(*) OVER (PARTITION BY wa.author_id, an_w.parsed_name.first) AS cluster_works
  FROM openalex.works.work_authors wa
  JOIN openalex.authors.author_names an_w ON wa.raw_author_name = an_w.raw_author_name
  WHERE wa.author_id IS NOT NULL
    AND an_w.parsed_name.last IS NOT NULL
    AND LENGTH(an_w.parsed_name.last) > 2
    AND LENGTH(an_w.parsed_name.first) > 2
    AND NOT REGEXP_LIKE(wa.raw_author_name, '[\uac00-\ud7af\u3040-\u30ff\u4e00-\u9fff]')
),
-- Per-row work raw_orcid from source metadata (used for the per-row ORCID exclusion).
work_orcids AS (
  SELECT wb.id AS work_id, pos AS author_sequence, authorship.raw_orcid
  FROM openalex.works.openalex_works_base wb
  LATERAL VIEW POSEXPLODE(wb.authorships) t AS pos, authorship
  WHERE authorship.raw_orcid IS NOT NULL AND authorship.raw_orcid != ''
),
-- Resolved institution IDs per (work_id, author_sequence). Source: work_author_affiliations
-- (BIGINT institution_id). Format aligned to authors_for_matching's URL form so they overlap.
-- Surfaced for the rollback cell, not used in candidate filter.
work_inst_ids AS (
  SELECT work_id, author_sequence,
    COLLECT_SET(CONCAT('https://openalex.org/I', CAST(institution_id AS STRING))) AS work_institutions
  FROM openalex.works.work_author_affiliations
  WHERE institution_id IS NOT NULL
  GROUP BY work_id, author_sequence
)
SELECT wp.work_id, wp.author_sequence, wp.author_id, wp.raw_author_name,
  wp.work_last, wp.work_first, wp.cluster_works,
  ap.last AS author_last, ap.first AS author_first, ap.full_name AS author_full_name,
  ap.orcid, ap.works_count,
  COALESCE(wi.work_institutions, ARRAY()) AS work_institutions,
  COALESCE(ap.institution_ids, ARRAY()) AS author_institution_ids
FROM work_parsed wp
JOIN openalex.authors.authors_for_matching ap ON wp.author_id = ap.author_id
LEFT JOIN work_inst_ids wi ON wp.work_id = wi.work_id AND wp.author_sequence = wi.author_sequence
WHERE wp.work_last = ap.last
  AND LENGTH(ap.last) > 2
  AND LENGTH(ap.first) > 2
  AND wp.work_first != ap.first
  AND INSTR(wp.work_first, ap.first) = 0
  AND INSTR(ap.first, wp.work_first) = 0
  AND INSTR(TRANSLATE(wp.work_first, '-', ''), TRANSLATE(ap.first, '-', '')) = 0
  AND INSTR(TRANSLATE(ap.first, '-', ''), TRANSLATE(wp.work_first, '-', '')) = 0
  AND wp.work_first NOT LIKE '%.%'
  AND ap.first NOT LIKE '%.%'
  AND wp.work_first NOT IN ('md','mohamed','mohammed','muhammad','lord','lady','sir','dame','ltcol','capt','col','rev','hon','prof','doctor')
  AND ap.first NOT IN ('md','mohamed','mohammed','muhammad','lord','lady','sir','dame','ltcol','capt','col','rev','hon','prof','doctor','&na')
  AND NOT REGEXP_LIKE(ap.full_name, '[\uac00-\ud7af\u3040-\u30ff\u4e00-\u9fff]')
  -- Per-row ORCID exclusion: drop only when the work's raw_orcid matches the profile's.
  -- Replaces the old profile-level ap.orcid IS NULL filter, allowing splits on
  -- ORCID-anchored profiles when individual outlier works lack matching raw_orcid.
  AND NOT EXISTS (
    SELECT 1 FROM work_orcids wo
    WHERE wo.work_id = wp.work_id
      AND wo.author_sequence = wp.author_sequence
      AND wo.raw_orcid = ap.orcid
  )
  -- Minority-cluster guard: only split when this first-name cluster is at most
  -- 30% of the profile's total works. Protects the profile's "main" identity.
  AND wp.cluster_works * 1.0 / NULLIF(ap.works_count, 0) <= 0.30

### Cell 2: Validation statistics

In [ ]:
SELECT COUNT(*) AS total_candidates,
  COUNT(DISTINCT author_id) AS distinct_authors,
  COUNT(DISTINCT work_id) AS distinct_works,
  PERCENTILE_APPROX(works_count, ARRAY(0.25, 0.5, 0.75, 0.95)) AS author_works_pctiles
FROM openalex.authors.overmerge_split_candidates_phase2

### Cell 3: Spot-check sample (manual review)

In [ ]:
SELECT work_id, author_id, raw_author_name, author_full_name,
  work_first, author_first, work_last,
  cluster_works, works_count AS author_works_count,
  ROUND(cluster_works * 1.0 / NULLIF(works_count, 0), 4) AS cluster_pct
FROM openalex.authors.overmerge_split_candidates_phase2
ORDER BY RAND()
LIMIT 50

### Cell 4: Create audit log (for rollback)

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.overmerge_split_log_phase2 AS
SELECT work_id, author_sequence, author_id AS previous_author_id,
  raw_author_name, work_first, author_first, work_last,
  current_timestamp() AS split_timestamp
FROM openalex.authors.overmerge_split_candidates_phase2

### Cell 5: Execute the split

**WARNING**: This nulls out author_ids. Verify cells 2-3 before running.

**NOTE**: MatchAuthors has cutoffs (`work_id > 7B`, `created_date >= 2025-12-20`). Older split records will NOT be re-processed automatically. A separate re-matching run or cutoff relaxation is needed (same issue as Phase 1).

In [ ]:
MERGE INTO openalex.works.work_authors AS target
USING (
  SELECT DISTINCT work_id, author_sequence
  FROM openalex.authors.overmerge_split_candidates_phase2
) AS source
ON target.work_id = source.work_id
   AND target.author_sequence = source.author_sequence
WHEN MATCHED THEN
  UPDATE SET
    target.author_id = NULL,
    target.updated_at = current_timestamp()

### Cell 6: Queue split works for reprocessing by UpdateWorkAuthorships

Push affected work_ids into `curated_work_ids_pending_sync` so the next `UpdateWorkAuthorships` run picks them up and propagates the nulled author_ids through to `work_authorships` → `openalex_works`.

In [ ]:
INSERT INTO openalex.institutions.curated_work_ids_pending_sync
SELECT DISTINCT work_id, NULL AS curated_ras, current_timestamp() AS added_datetime
FROM openalex.authors.overmerge_split_candidates_phase2

### Cell 7: Post-split verification

In [ ]:
SELECT
  COUNT(*) AS nulled_records,
  COUNT(DISTINCT c.work_id) AS works_affected
FROM openalex.works.work_authors wa
JOIN openalex.authors.overmerge_split_log_phase2 c
  ON wa.work_id = c.work_id AND wa.author_sequence = c.author_sequence
WHERE wa.author_id IS NULL

### Cell 8: Outcome tracking (run after MatchAuthors re-processes)

In [ ]:
SELECT
  CASE
    WHEN wa.author_id IS NULL THEN 'STILL_UNMATCHED'
    WHEN wa.author_id = log.previous_author_id THEN 'RE_MATCHED_SAME'
    ELSE 'RE_MATCHED_NEW'
  END AS outcome,
  COUNT(*) AS cnt
FROM openalex.works.work_authors wa
JOIN openalex.authors.overmerge_split_log_phase2 log
  ON wa.work_id = log.work_id AND wa.author_sequence = log.author_sequence
GROUP BY 1

### Cell 9: Rollback first-name-evolution false positives

Phase 2 incorrectly splits works for authors whose first name legitimately evolved over time (e.g., transgender name change, anglicization, formal-vs-nickname like Jonathan ↔ Jon). Last name + same person + different first name is the classic shape. When the work shares an institution with the author profile, the differing first name likely reflects a name evolution, not an overmerge.

**Co-signal requirement:** restore only when work shares any institution with the author profile's affiliations (`work_institutions ∩ author_institution_ids`).

**Note:** ORCID match isn't checked here because Cell 1 already excludes any candidate where the work's `raw_orcid` matches the author profile's ORCID — by construction it can never fire as a positive signal at this stage.

Only updates rows still NULL (won't overwrite re-matches).

In [ ]:
MERGE INTO openalex.works.work_authors AS target
USING (
  SELECT work_id, author_sequence, FIRST(author_id) AS previous_author_id
  FROM openalex.authors.overmerge_split_candidates_phase2
  WHERE ARRAYS_OVERLAP(work_institutions, author_institution_ids)
  GROUP BY work_id, author_sequence
) AS source
ON target.work_id = source.work_id
   AND target.author_sequence = source.author_sequence
WHEN MATCHED AND target.author_id IS NULL THEN
  UPDATE SET
    target.author_id = source.previous_author_id,
    target.updated_at = current_timestamp()

### Cell 10: Queue restored works for reprocessing

In [ ]:
INSERT INTO openalex.institutions.curated_work_ids_pending_sync
SELECT DISTINCT work_id, NULL AS curated_ras, current_timestamp() AS added_datetime
FROM openalex.authors.overmerge_split_candidates_phase2
WHERE ARRAYS_OVERLAP(work_institutions, author_institution_ids)

### Cell 11: Post-rollback verification

In [ ]:
SELECT
  COUNT(*) AS restored_records,
  COUNT(DISTINCT c.author_id) AS restored_authors
FROM openalex.works.work_authors wa
JOIN openalex.authors.overmerge_split_candidates_phase2 c
  ON wa.work_id = c.work_id AND wa.author_sequence = c.author_sequence
WHERE ARRAYS_OVERLAP(c.work_institutions, c.author_institution_ids)
  AND wa.author_id = c.author_id